# Sentence Vector Search Demo

This notebook demonstrates semantic search across Latin texts using sentence vectors.

The `SentenceVectorStore` indexes sentences by their mean-of-token vectors (from LatinCy),
then uses cosine similarity for fast retrieval. No external vector DB required — just NumPy.

In [1]:
from latincyreaders import TesseraeReader
from latincyreaders.cache.vectors import SentenceVectorConfig, SentenceVectorStore

## 1. Set up reader and pick files to index

In [2]:
reader = TesseraeReader()
print(f"Corpus: {len(reader.fileids())} files")

Corpus: 900 files


In [3]:
# Pick a subset for the demo — a few major authors
demo_files = [
    f for f in reader.fileids()
    if any(a in f for a in ['vergil', 'cicero.de_amicitia', 'caesar.de_bello_gallico', 'ovid.metamorphoses'])
]
print(f"Demo files: {len(demo_files)}")
for f in demo_files[:10]:
    print(f"  {f}")
if len(demo_files) > 10:
    print(f"  ... and {len(demo_files) - 10} more")

Demo files: 41
  caesar.de_bello_gallico.part.1.tess
  caesar.de_bello_gallico.part.2.tess
  caesar.de_bello_gallico.part.3.tess
  caesar.de_bello_gallico.part.4.tess
  caesar.de_bello_gallico.part.5.tess
  caesar.de_bello_gallico.part.6.tess
  caesar.de_bello_gallico.part.7.tess
  caesar.de_bello_gallico.part.8.tess
  cicero.de_amicitia.tess
  ovid.metamorphoses.part.1.tess
  ... and 31 more


## 2. Build the vector index

This iterates through all selected files, runs them through the LatinCy pipeline,
computes mean-of-token vectors per sentence, and saves to disk as a memory-mapped `.npy` file.

In [4]:
import time

cfg = SentenceVectorConfig(collection="demo")
store = SentenceVectorStore(cfg)

t0 = time.perf_counter()
count = store.build(reader, demo_files)
elapsed = time.perf_counter() - t0

stats = store.stats()
print(f"Indexed {count:,} sentences in {elapsed:.1f}s")
print(f"Vector dim: {stats['vector_dim']}")
print(f"Index size: {stats['size_bytes'] / 1024:.0f} KB")

Indexed 15,800 sentences in 59.9s
Vector dim: 300
Index size: 21565 KB


## 3. Semantic search

Query with any Latin text — the model vectorizes it and finds the closest sentences by cosine similarity.

In [5]:
nlp = reader.nlp

def search(query, top_k=5):
    """Pretty-print semantic search results."""
    results = store.similar_to_sent(query, nlp, top_k=top_k)
    print(f"Query: {query!r}\n")
    for i, r in enumerate(results, 1):
        citation = r.get('citation', '') or r['fileid']
        print(f"  {i}. [{r['score']:.3f}] {citation}")
        print(f"     {r['text'][:120]}")
    print()

In [6]:
search("arma virumque cano")

Query: 'arma virumque cano'

  1. [0.796] <ov. met. 8.21>
     Iamque mora belli procerum quoque nomina norat armaque equosque habitusque Cydonaeasque pharetras.
  2. [0.761] <verg. aen. 9.777>
     [Semper equos atque arma virum pugnasque canebat.
  3. [0.757] <verg. g. 3.343>
     Omnia secum armentarius Afer agit, tectumque laremque armaque Amyclaeumque canem Cressamque pharetram;
  4. [0.755] <verg. aen. 11.746>
     Volat igneus aequore Tarchon arma virumque ferens;
  5. [0.735] <verg. aen. 7.460>
     arma amens fremit, arma toro tectisque requirit;



In [7]:
search("amicitia")

Query: 'amicitia'

  1. [0.516] <cic. amicit. 51>
     non igitur utilitatem amicitia, sed utilitas amicitiam secuta est.
  2. [0.482] <cic. amicit. 26>
     quapropter a natura mihi videtur potius quam indigentia orta amicitia, applicatione magis animi cum quodam sensu amandi,
  3. [0.473] <cic. amicit. 96>
     ita re magis quam summa auctoritate causa illa defensa est.
  4. [0.465] <cic. amicit. 4>
     cum enim saepe mecum ageres, ut de amicitia scriberem aliquid, digna mihi res cum omnium cognitione tum nostra familiari
  5. [0.450] <cic. amicit. 74>
     disparis enim mores disparia studia sequuntur, quorum dissimilitudo dissociat amicitias;



In [8]:
search("Gallia est omnis divisa in partes tres")

Query: 'Gallia est omnis divisa in partes tres'

  1. [0.695] <caes. gal. 1.12.4>
     nam omnis civitas Helvetia in quattuor pagos divisa est.
  2. [0.639] <caes. gal. 1.1.1>
     Gallia est omnis divisa in partes tres, quarum unam incolunt Belgae, aliam Aquitani, tertiam qui ipsorum lingua Celtae, 
  3. [0.627] <caes. gal. 6.11.5>
     namque omnes civitates in partes divisae sunt duas.
  4. [0.612] <caes. gal. 6.44.3>
     Quibus cum aqua atque igni interdixisset, duas legiones ad fines Treverorum, duas in Lingonibus, sex reliquas in Senonum
  5. [0.608] <caes. gal. 8.46.4>
     quattuor legiones in Belgio collocavit cum M. Antonio et C. Trebonio et P. Vatinio legatis, duas legiones in Aeduos dedu



In [9]:
search("amor")

Query: 'amor'

  1. [0.598] <verg. ecl. 2.68>
     me tamen urit amor;
  2. [0.545] <ov. met. 10.315>
     hic amor est odio maius scelus.
  3. [0.545] <ov. met. 7.800>
     Mutua cura duos et amor socialis habebat;
  4. [0.545] <ov. met. 14.633>
     hic amor, hoc studium;
  5. [0.537] <ov. met. 3.392>
     Sed tamen haeret amor crescitque dolore repulsae.



In [10]:
search("ira deorum")

Query: 'ira deorum'

  1. [0.668] <verg. aen. 7.461>
     saevit amor ferri et scelerata insania belli, ira super:
  2. [0.573] <ov. met. 6.622>
     Nec plura locuta triste parat facinus tacitaque exaestuat ira.
  3. [0.568] <verg. aen. 11.232>
     Fatalem Aenean manifesto numine ferri admonet ira deum tumulique ante ora recentes.
  4. [0.565] <ov. met. 8.279>
     Tangit et ira deos.
  5. [0.563] <verg. aen. 2.574>
     subit ira cadentem ulcisci patriam et sceleratas sumere poenas.



## 4. Reader shortcut

`reader.find_similar()` wraps the store for convenience. With `auto_build=True`,
it will build the index automatically if one doesn't exist yet.

In [11]:
results = reader.find_similar("virtus et gloria", top_k=5, config=cfg)
for r in results:
    citation = r.get('citation', '') or r['fileid']
    print(f"[{r['score']:.3f}] {citation}")
    print(f"  {r['text'][:120]}\n")

[0.647] <ov. met. 15.202>
  tunc herba nitens et roboris expers turget et insolida est et spe delectat agrestes.

[0.641] <verg. aen. 5.343>
  Tutatur favor Euryalum, lacrimaeque decorae, gratior et pulchro veniens in corpore virtus.

[0.635] <cic. amicit. 5>
  nunc Laelius et sapiens, sic enim est habitus, et amicitiae gloria excellens de amicitia loquetur.

[0.604] <cic. amicit. 88>
  una illa subeunda est offensio ut et utilitas in amicitia et fides retineatur;

[0.594] <caes. gal. 5.34.2>
  Erant et virtute et studio pugnandi pares;



## 5. Search by existing sentence

Find sentences similar to one already in the index — no need to re-vectorize.

In [12]:
# Find sentences similar to the first sentence of Cicero's De Amicitia
results = store.similar_to_doc_sent("cicero.de_amicitia.tess", 0, top_k=5)
for r in results:
    citation = r.get('citation', '') or r['fileid']
    print(f"[{r['score']:.3f}] {citation}")
    print(f"  {r['text'][:120]}\n")

[1.000] <cic. amicit. 1>
  Q. Mucius augur multa narrare de C. Laelio socero suo memoriter et iucunde solebat nec dubitare illum in omni sermone ap

[0.687] <cic. amicit. 96>
  atque, ut ad me redeam, meministis Q. Maximo fratre Scipionis et L. Mancino consulibus, quam popularis lex de sacerdotii

[0.664] <cic. amicit. 3>
  itaque tum Scaevola, cum in eam ipsam mentionem incidisset, exposuit nobis sermonem Laeli de amicitia habitum ab illo se

[0.661] <caes. gal. 1.21.4>
  P. Considius, qui rei militaris peritissimus habebatur et in exercitu L. Sullae et postea in M. Crassi fuerat, cum explo

[0.650] <cic. amicit. 95>
  quibus blanditiis C. Papirius nuper influebat in auris contionis, cum ferret legem de tribunis plebis reficiendis!



## 6. Index statistics

In [13]:
store.stats()

{'collection': 'demo',
 'sentences': 15800,
 'vector_dim': 300,
 'size_bytes': 22083035,
 'store_dir': '/Users/pjb311/latincy_data/vectors/demo'}

## Cleanup

Uncomment to remove the demo index from disk.

In [14]:
# store.clear()
# print("Index cleared.")